# Feature engineering - data preparation pipeline  | Sebislaw

## Libraries

In [1]:
from os.path  import join
import random
import itertools
import math

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
import xgboost as xgb
from xgboost import XGBClassifier
import optuna

## Data

In [2]:
data_path = '..\\..\\..\\data'

# The Basics ------------------------------------------------------------------------
# Men
MTeams = pd.read_csv(join(data_path, 'MTeams.csv'))
MSeasons = pd.read_csv(join(data_path, 'MSeasons.csv'))
MNCAATourneySeeds = pd.read_csv(join(data_path, 'MNCAATourneySeeds.csv'))
MRegularSeasonCompactResults = pd.read_csv(join(data_path, 'MRegularSeasonCompactResults.csv'))
MNCAATourneyCompactResults = pd.read_csv(join(data_path, 'MNCAATourneyCompactResults.csv'))
# Women
WTeams = pd.read_csv(join(data_path, 'WTeams.csv'))
WSeasons = pd.read_csv(join(data_path, 'WSeasons.csv'))
WNCAATourneySeeds = pd.read_csv(join(data_path, 'WNCAATourneySeeds.csv'))
WRegularSeasonCompactResults = pd.read_csv(join(data_path, 'WRegularSeasonCompactResults.csv'))
WNCAATourneyCompactResults = pd.read_csv(join(data_path, 'WNCAATourneyCompactResults.csv'))
# Other
SampleSubmissionStage1 = pd.read_csv(join(data_path, 'SampleSubmissionStage1.csv'))
SampleSubmissionStage2 = pd.read_csv(join(data_path, 'SampleSubmissionStage2.csv'))
SeedBenchmarkStage1 = pd.read_csv(join(data_path, 'SeedBenchmarkStage1.csv'))

# Team Box Scores ------------------------------------------------------------------------
# Men
MRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'MRegularSeasonDetailedResults.csv'))
MNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'MNCAATourneyDetailedResults.csv'))
# Women
WRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'WRegularSeasonDetailedResults.csv'))
WNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'WNCAATourneyDetailedResults.csv'))

# Geography ------------------------------------------------------------------------
# All
Cities = pd.read_csv(join(data_path, 'Cities.csv'))
Conferences = pd.read_csv(join(data_path, 'Conferences.csv'))
# Men
MGameCities = pd.read_csv(join(data_path, 'MGameCities.csv'))
# Women
WGameCities = pd.read_csv(join(data_path, 'WGameCities.csv'))

# Public Rankings ------------------------------------------------------------------------
# Men
MMasseyOrdinals = pd.read_csv(join(data_path, 'MMasseyOrdinals.csv')) # men only

# Supplements ------------------------------------------------------------------------
# Men
MTeamCoaches = pd.read_csv(join(data_path, 'MTeamCoaches.csv')) # men only
MTeamConferences = pd.read_csv(join(data_path, 'MTeamConferences.csv'))
MConferenceTourneyGames = pd.read_csv(join(data_path, 'MConferenceTourneyGames.csv'))
MSecondaryTourneyTeams = pd.read_csv(join(data_path, 'MSecondaryTourneyTeams.csv'))
MSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'MSecondaryTourneyCompactResults.csv'))
MTeamSpellings = pd.read_csv(join(data_path, "MTeamSpellings.csv"), encoding='cp1252')
MNCAATourneySlots = pd.read_csv(join(data_path, 'MNCAATourneySlots.csv'))
MNCAATourneySeedRoundSlots = pd.read_csv(join(data_path, 'MNCAATourneySeedRoundSlots.csv')) # men only
# Women
WTeamConferences = pd.read_csv(join(data_path, 'WTeamConferences.csv'))
WConferenceTourneyGames = pd.read_csv(join(data_path, 'WConferenceTourneyGames.csv'))
WSecondaryTourneyTeams = pd.read_csv(join(data_path, 'WSecondaryTourneyTeams.csv'))
WSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'WSecondaryTourneyCompactResults.csv'))
WTeamSpellings = pd.read_csv(join(data_path, 'WTeamSpellings.csv'), encoding='cp1252')
WNCAATourneySlots = pd.read_csv(join(data_path, 'WNCAATourneySlots.csv'))

FileNotFoundError: [Errno 2] No such file or directory: '..\\..\\..\\data\\MTeams.csv'

## Data preparation pipeline

In [3]:
def prepare_data(df):
        
    """
    This function duplicates and flips a game record.
    Now two records with the same data are present, 
    but viewed from perspectives of two different teams.
    """
    
    df = df[[
         'Season', 'DayNum', 'NumOT',
         'WTeamID',  'WScore', 'WLoc',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
         'LTeamID', 'LScore',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'
    ]]
    dfswap = df[[
         'Season', 'DayNum', 'NumOT',
         'LTeamID', 'LScore', 'WLoc',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF',
         'WTeamID',  'WScore',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
    ]].copy()
    
    dfswap.loc[df['WLoc'] == 'H', 'WLoc'] = 'A'
    dfswap.loc[df['WLoc'] == 'A', 'WLoc'] = 'H'
        
    df = df.rename(columns={'WLoc': 'location'})
    dfswap = dfswap.rename(columns={'WLoc': 'location'})
        
    df.columns = [x.replace('W','T1_').replace('L','T2_') for x in list(df.columns)]
    dfswap.columns = [x.replace('L','T1_').replace('W','T2_') for x in list(dfswap.columns)]
    
    output = pd.concat([df, dfswap]).reset_index(drop=True)
    output.loc[output.location=='N','location'] = '0'
    output.loc[output.location=='H','location'] = '1'
    output.loc[output.location=='A','location'] = '-1'
    output.location = output.location.astype(int)
        
    output['PointDiff'] = output['T1_Score'] - output['T2_Score']
    
    return output

def get_data(regular_results, tourney_results, seeds, prepared=False):

    if prepared:
        regular_data = regular_results.copy()
        tourney_data = tourney_results.copy()
    else:
        # make data frames with extra rows to represent the perspective of losing team
        regular_data = prepare_data(regular_results)
        tourney_data  = prepare_data(tourney_results)
    
    # data frame with mean game statistics for a given team in a given season
    season_statistics  = regular_data.groupby(["Season", 'T1_TeamID'])[
        [
            'T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA','T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF',
            'T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA','T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF',
            'PointDiff'
        ]
    ].agg('mean').reset_index()
    
    # mean statistics for team and team's opponent's
    season_statistics_T1 = season_statistics .copy()
    season_statistics_T2 = season_statistics .copy()
    
    season_statistics_T1.columns = ["T1_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T1.columns)]
    season_statistics_T2.columns = ["T2_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T2.columns)]
    season_statistics_T1.columns.values[0] = "Season"
    season_statistics_T2.columns.values[0] = "Season"
    
    # data frame containing game's result
    tourney_data = tourney_data[['Season', 'DayNum', 'T1_TeamID', 'T1_Score', 'T2_TeamID' ,'T2_Score', 'location']]
    tourney_data = pd.merge(tourney_data, season_statistics_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, season_statistics_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # data frame with win fraction from last x days for a given team in a given season
    last14days_stats_T1 = regular_data.loc[regular_data.DayNum>118].reset_index(drop=True)
    last14days_stats_T1['win'] = np.where(last14days_stats_T1['PointDiff']>0,1,0)
    last14days_stats_T1 = last14days_stats_T1.groupby(['Season','T1_TeamID'])['win'].mean().reset_index(name='T1_win_ratio_14d')
    
    last14days_stats_T2 = regular_data.loc[regular_data.DayNum>118].reset_index(drop=True)
    last14days_stats_T2['win'] = np.where(last14days_stats_T2['PointDiff']<0,1,0)
    last14days_stats_T2 = last14days_stats_T2.groupby(['Season','T2_TeamID'])['win'].mean().reset_index(name='T2_win_ratio_14d')
    
    # add to tourney_data column with win fraction for winning and losing team
    tourney_data = pd.merge(tourney_data, last14days_stats_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, last14days_stats_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # get seeds with no regional division
    seeds['seed'] = seeds['Seed'].apply(lambda x: int(x[1:3]))
    
    # give each team a raw seed
    seeds_T1 = seeds[['Season','TeamID','seed']].copy()
    seeds_T2 = seeds[['Season','TeamID','seed']].copy()
    seeds_T1.columns = ['Season','T1_TeamID','T1_seed']
    seeds_T2.columns = ['Season','T2_TeamID','T2_seed']
    
    # add seeds to turney data for team 1 and team 2
    tourney_data = pd.merge(tourney_data, seeds_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, seeds_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # add a seed difference column
    tourney_data["Seed_diff"] = tourney_data["T1_seed"] - tourney_data["T2_seed"]

    return tourney_data

def get_x_y(seeds,
            season_games, season_range, days_back,
            tourney_games, tourney_range, 
            other_parameters=0):
    
    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]
    
    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Get final data frame
    df = get_data(regular_results, tourney_results, seeds)
    
    # Prepare data and labels
    x = df[list(df.columns[7:999])].values
    y = np.where(df['T1_Score'] - df['T2_Score'] > 0, 1, 0)
    
    return df, x, y

def get_final_data(seeds,
                   season_games, season_range, days_back,
                   tourney_games, tourney_range, 
                   SampleSubmissionStage1):

    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]

    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Assume sample_submission is a DataFrame with an "ID" column like "2023_1101_1102"
    # and tourney_results is a DataFrame with columns including: Season, WTeamID, LTeamID, DayNum, WScore, LScore, WLoc, etc.
    # Filter rows where the ID starts with the specified season (followed by an underscore)
    final_season = tourney_range[0]
    sample_submission_copy = SampleSubmissionStage1.copy()
    sample_submission = sample_submission_copy[sample_submission_copy['ID'].str.startswith(f"{final_season}_")]
    sample_submission = sample_submission.drop(columns=['Pred'])
    
    sample_submission[['Season', 'Team1', 'Team2']] = sample_submission['ID'].str.split('_', expand=True)
    sample_submission['Season'] = sample_submission['Season'].astype(int)
    sample_submission['Team1'] = sample_submission['Team1'].astype(int)
    sample_submission['Team2'] = sample_submission['Team2'].astype(int)
    
    regular_data_final = prepare_data(regular_results)
    tourney_data_final  = prepare_data(tourney_results)
    
    # Merge the two DataFrames using a left join on the keys:
    # # For sample_submission: keys are Season, Team1, Team2.
    # # For regular_data_final: keys are Season, T1_TeamID, T2_TeamID.
    # final_regular_season_with_rows_as_in_submission = pd.merge(
    #     sample_submission,
    #     regular_data_final,
    #     left_on=['Season', 'Team1', 'Team2'],
    #     right_on=['Season', 'T1_TeamID', 'T2_TeamID'],
    #     how='left'
    # )
    # final_tourney_data_with_rows_as_in_submission = pd.merge(
    #     sample_submission,
    #     tourney_data_final,
    #     left_on=['Season', 'Team1', 'Team2'],
    #     right_on=['Season', 'T1_TeamID', 'T2_TeamID'],
    #     how='left'
    # )

    # tourney_data =  get_data(final_regular_season_with_rows_as_in_submission, final_tourney_data_with_rows_as_in_submission,
    #                          seeds, prepared=True)

    tourney_data =  get_data(regular_data_final, tourney_data_final,
                             seeds, prepared=True)

    # Prepare data and labels
    x = tourney_data[list(tourney_data.columns[7:999])].values
    y = np.where(
        tourney_data[['T1_Score', 'T2_Score']].isnull().any(axis=1),
        np.nan,
        np.where(tourney_data['T1_Score'] - tourney_data['T2_Score'] > 0, 1, 0)
    )

    return tourney_data, x, y

## Prepare train and test data set

In [4]:
regular_results = pd.concat([
    MRegularSeasonDetailedResults.copy(),
    WRegularSeasonDetailedResults.copy()
], ignore_index=True)
tourney_results = pd.concat([
    MNCAATourneyDetailedResults.copy(),
    WNCAATourneyDetailedResults.copy()
], ignore_index=True)
seeds = pd.concat([
    MNCAATourneySeeds.copy(),
    WNCAATourneySeeds.copy()
], ignore_index=True)

# PARAMETERS TO CHANGE
# ----------------------------------------------------------
final_season = 2023
start_season = 2005
season_years_list  = [[i-2, i-1, i] for i in range(start_season, final_season+1)]
days_back = 30
# ----------------------------------------------------------

# This ensures that we simulate the scenarion in competition
tourney_results_final = tourney_results[tourney_results['Season'] == final_season]
tourney_results = tourney_results[tourney_results['Season'] < final_season]
tourney_years_list = [[i] for i in range(start_season, final_season+1)]

data = []
x = []
y = []
data_final = []
x_final_season = []
y_final_season = []
for season_years, tourney_years in zip(season_years_list, tourney_years_list):
    if tourney_years[0] != 2020:
        if tourney_years[0] == final_season:
            data_tmp_final, x_tmp, y_tmp = get_final_data(seeds,
                   regular_results, season_years, days_back,
                   tourney_results_final, tourney_years, 
                   SampleSubmissionStage1)
            data_final.append(data_tmp_final)
            x_final_season.append(x_tmp)
            y_final_season.append(y_tmp)
        else:
            data_tmp, x_tmp, y_tmp = get_x_y(
                seeds,
                regular_results, season_years, days_back,
                tourney_results, tourney_years)
            data.append(data_tmp)
            x.append(x_tmp)
            y.append(y_tmp)

# Concatenate numpy arrays along the first axis
x = np.concatenate(x, axis=0)
y = np.concatenate(y, axis=0)
x_final_season = np.concatenate(x_final_season, axis=0)
y_final_season = np.concatenate(y_final_season, axis=0)
        
# Split data frame into train and test
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)
df = pd.concat(data, ignore_index=True)
df_final = pd.concat(data_final, ignore_index=True)

## First models

In [5]:
# Create a pipeline with StandardScaler and LogisticRegressionCV
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegressionCV(
        cv=5,                # 5-fold cross-validation
        solver='lbfgs',
        max_iter=2000,       # Increase max_iter if needed
        scoring='neg_log_loss',  # Optimize log-loss for probability calibration
        refit=True
    ))
])

# Train the model using the pipeline
pipeline.fit(x, y)

# Predict probabilities for the positive class
# Create masks for training data:
mask_train = ~np.isnan(x).any(axis=1) & ~np.isnan(y)
x_clean = x[mask_train]
y_clean = y[mask_train]
# And for the final season (test) data:
mask_test = ~np.isnan(x_final_season).any(axis=1) & ~np.isnan(y_final_season)
x_final_clean = x_final_season[mask_test]
y_final_clean = y_final_season[mask_test]
# Predict probabilities for the positive class on the cleaned final season data
y_prob = pipeline.predict_proba(x_final_clean)[:, 1]

# Calculate the Brier score using the cleaned test labels
score = brier_score_loss(y_final_clean, y_prob)
print("Brier score:", score)

Brier score: 0.17882030796019963


In [6]:
# Create and train the XGBoost classifier
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
model.fit(x, y)

# Predict probabilities for the positive class
# Create masks for training data:
mask_train = ~np.isnan(x).any(axis=1) & ~np.isnan(y)
x_clean = x[mask_train]
y_clean = y[mask_train]
# And for the final season (test) data:
mask_test = ~np.isnan(x_final_season).any(axis=1) & ~np.isnan(y_final_season)
x_final_clean = x_final_season[mask_test]
y_final_clean = y_final_season[mask_test]
# Predict probabilities for the positive class on the cleaned final season data
y_prob = model.predict_proba(x_final_clean)[:, 1]

# Calculate the Brier score
brier = brier_score_loss(y_final_clean, y_prob)
print("Brier Score:", brier)

Brier Score: 0.21200261550085395
